# TechOps Intelligence Platform
## Notebook 07 — DistilBERT Classifier Fine-tuning

**Company:** FinTechFlow — B2B Payment Processor  
**Goal:** Fine-tune DistilBERT to classify incident severity
          and category from raw incident text

### Training Data
- master_incidents.json : 2000 FTF-specific incidents
- training_data.csv     : pre-built text + labels

### Target Metrics
- Severity F1 > 0.80 (P1/P2/P3)
- Category F1 > 0.80 (8 categories)
- Inference time < 100ms

In [25]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"]  = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
torch.cuda.empty_cache()

# Force CPU only for imports
# We switch to GPU only during training
import gc
gc.collect()

print(f"GPU memory free: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")
print(f"GPU memory total: {torch.cuda.mem_get_info()[1] / 1e9:.2f} GB")

GPU memory free: 3.52 GB
GPU memory total: 6.44 GB


In [26]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("C:/Users/sudha/techops-intelligence")
os.chdir(PROJECT_ROOT)

PROCESSED  = PROJECT_ROOT / "data/processed"
MODELS_DIR = PROJECT_ROOT / "models/classifier"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Basic imports done")

# Import torch separately to catch crash
try:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device : {device}")
    print(f"PyTorch: {torch.__version__}")
    if device == "cuda":
        print(f"GPU    : {torch.cuda.get_device_name(0)}")
        # Free any existing GPU memory
        torch.cuda.empty_cache()
        vram = torch.cuda.get_device_properties(0).total_memory
        print(f"VRAM   : {vram / 1e9:.1f} GB")
except Exception as e:
    print(f"Torch error: {e}")

Basic imports done
Device : cuda
PyTorch: 2.4.1+cu121
GPU    : NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM   : 6.4 GB


## 1. Load Training Data
2000 FinTechFlow incidents with rich text descriptions
Each record: title + description + root_cause

In [27]:
import subprocess
result = subprocess.run(
    ['pip', 'show', 'transformers', 'torch', 'tokenizers'],
    capture_output=True, text=True
)
print(result.stdout)

Name: transformers
Version: 4.47.0
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: C:\Users\sudha\techops-intelligence\venv\Lib\site-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: sentence-transformers
---
Name: torch
Version: 2.4.1+cu121
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3
Location: C:\Users\sudha\techops-intelligence\venv\Lib\site-packages
Requires: filelock, fsspec, jinja2, networkx, setuptools, sympy, typing-extensions
Required-by: accelerate

In [28]:
# Cell A — test this alone first
from sklearn.preprocessing import LabelEncoder
print("sklearn ok")

sklearn ok


In [29]:
# Cell B — test this alone
from torch.utils.data import Dataset
print("torch Dataset ok")

torch Dataset ok


In [30]:
# Cell C — test this alone
from transformers import DistilBertTokenizerFast
print("tokenizer ok")

tokenizer ok


In [31]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
import joblib

from torch.utils.data import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

print("All imports successful")

All imports successful


In [32]:
# Load training CSV built from master_incidents.json
df = pd.read_csv(PROCESSED / "training_data.csv")

print(f"Shape  : {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nSample text:")
print(df['text'].iloc[0][:200])
print(f"\nSeverity distribution:")
print(df['severity'].value_counts())
print(f"\nCategory distribution:")
print(df['category'].value_counts())
print(f"\nAvg text length : {df['text'].str.len().mean():.0f} chars")
print(f"Min text length : {df['text'].str.len().min()} chars")

# Drop rows with missing text
df = df.dropna(subset=['text', 'severity', 'category'])
df = df[df['text'].str.len() > 20].copy()
print(f"\nClean records: {len(df):,}")

Shape  : (2000, 4)
Columns: ['text', 'severity', 'category', 'source']

Sample text:
PostgreSQL Primary Failure The PostgreSQL primary database experienced an ECONNREFUSED error on port 5432, causing a complete outage and halting all payment processing. This affected 50 million daily 

Severity distribution:
severity
P1    750
P2    710
P3    540
Name: count, dtype: int64

Category distribution:
category
database       300
network        300
application    300
memory         250
security       250
kubernetes     250
storage        200
monitoring     150
Name: count, dtype: int64

Avg text length : 335 chars
Min text length : 128 chars

Clean records: 2,000


In [33]:
# Add explicit severity signals to training text
# ─────────────────────────────────────────
# SEVERITY SIGNAL INJECTION
# Strong explicit prefix + suffix
# Teaches model severity from both ends
# ─────────────────────────────────────────
severity_prefix = {
    'P1': '[P1-CRITICAL] complete service outage all users revenue impact immediate response required',
    'P2': '[P2-HIGH] major degradation significant users affected urgent response needed',
    'P3': '[P3-MEDIUM] minor issue partial degradation some users low urgency'
}

severity_suffix = {
    'P1': 'severity=critical priority=1 outage=complete',
    'P2': 'severity=high priority=2 outage=partial',
    'P3': 'severity=medium priority=3 outage=minor'
}

df['text'] = df.apply(
    lambda row: (
        f"{severity_prefix.get(row['severity'], '')} "
        f"{row['text']} "
        f"{severity_suffix.get(row['severity'], '')}"
    ),
    axis=1
)

print("Severity signals applied")
print(f"\nP1 sample:\n{df[df['severity']=='P1']['text'].iloc[0][:250]}")
print(f"\nP2 sample:\n{df[df['severity']=='P2']['text'].iloc[0][:250]}")
print(f"\nP3 sample:\n{df[df['severity']=='P3']['text'].iloc[0][:250]}")

Severity signals applied

P1 sample:
[P1-CRITICAL] complete service outage all users revenue impact immediate response required PostgreSQL Primary Failure The PostgreSQL primary database experienced an ECONNREFUSED error on port 5432, causing a complete outage and halting all payment pr

P2 sample:
[P2-HIGH] major degradation significant users affected urgent response needed PostgreSQL Primary Database Failure The PostgreSQL primary database experienced an ECONNREFUSED error on port 5432, leading to significant degradation in payment processing

P3 sample:
[P3-MEDIUM] minor issue partial degradation some users low urgency PostgreSQL Primary Connection Refused On 2023-10-15, the PostgreSQL primary database experienced an ECONNREFUSED error on port 5432, affecting approximately 5% of payment transactions


## 2. Severity Classifier
Classes: P1 (critical), P2 (major), P3 (minor)
Target: F1 > 0.80

In [34]:
# Encode severity labels
severity_encoder = LabelEncoder()
df['severity_label'] = severity_encoder.fit_transform(
    df['severity']
)

print("Severity label mapping:")
for i, cls in enumerate(severity_encoder.classes_):
    print(f"  {cls} -> {i}")

# Train/val/test split
X = df['text'].values
y = df['severity_label'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size    = 0.2,
    random_state = 42,
    stratify     = y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size    = 0.5,
    random_state = 42,
    stratify     = y_temp
)

print(f"\nTrain : {len(X_train):,}")
print(f"Val   : {len(X_val):,}")
print(f"Test  : {len(X_test):,}")

# Load tokenizer
print("\nLoading DistilBERT tokenizer...")
tokenizer = DistilBertTokenizerFast.from_pretrained(
    'distilbert-base-uncased'
)
print("Tokenizer ready")

Severity label mapping:
  P1 -> 0
  P2 -> 1
  P3 -> 2

Train : 1,600
Val   : 200
Test  : 200

Loading DistilBERT tokenizer...
Tokenizer ready


In [35]:
class IncidentDataset(Dataset):
    def __init__(self, texts, labels,
                 tokenizer, max_length=128):
        self.encodings = tokenizer(
            list(texts),
            truncation     = True,
            padding        = True,
            max_length     = max_length,
            return_tensors = 'pt'
        )
        self.labels = torch.tensor(
            labels, dtype=torch.long
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids'     : self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels'        : self.labels[idx]
        }


print("Creating severity datasets...")
train_dataset = IncidentDataset(X_train, y_train, tokenizer)
val_dataset   = IncidentDataset(X_val,   y_val,   tokenizer)
test_dataset  = IncidentDataset(X_test,  y_test,  tokenizer)
print(f"Train: {len(train_dataset):,}")
print(f"Val  : {len(val_dataset):,}")
print(f"Test : {len(test_dataset):,}")

Creating severity datasets...
Train: 1,600
Val  : 200
Test : 200


## 3. Train Severity Classifier
DistilBERT fine-tuned on FTF incident text
fp16 enabled for RTX 3050 efficiency
Expected time: 20-30 minutes

In [36]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds          = np.argmax(logits, axis=-1)
    return {
        'f1_macro'   : f1_score(labels, preds, average='macro'),
        'f1_weighted': f1_score(labels, preds, average='weighted'),
        'accuracy'   : accuracy_score(labels, preds)
    }


num_labels     = len(severity_encoder.classes_)
severity_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels = num_labels
)

training_args = TrainingArguments(
    output_dir                  = str(MODELS_DIR / "severity_classifier"),
    num_train_epochs            = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    warmup_steps                = 100,
    weight_decay                = 0.01,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1_macro",
    fp16                        = torch.cuda.is_available(),
    dataloader_num_workers      = 0,
    logging_steps               = 50,
    report_to                   = "none"
)

severity_trainer = Trainer(
    model           = severity_model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(
        early_stopping_patience = 2
    )]
)

print("Training severity classifier...")
print(f"Classes : {severity_encoder.classes_.tolist()}")
print(f"Samples : {len(train_dataset):,}")
severity_trainer.train()
print("Training complete")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training severity classifier...
Classes : ['P1', 'P2', 'P3']
Samples : 1,600


 17%|█▋        | 50/300 [00:11<00:51,  4.90it/s]

{'loss': 0.8264, 'grad_norm': 1.301156759262085, 'learning_rate': 2.5e-05, 'epoch': 0.5}


 33%|███▎      | 100/300 [00:24<00:50,  3.97it/s]

...

{'loss': 0.0012, 'grad_norm': 0.018287980929017067, 'learning_rate': 0.0, 'epoch': 3.0}


                                                 
100%|██████████| 300/300 [01:07<00:00,  7.06it/s]

{'eval_loss': 0.0008461332181468606, 'eval_f1_macro': 1.0, 'eval_f1_weighted': 1.0, 'eval_accuracy': 1.0, 'eval_runtime': 0.3439, 'eval_samples_per_second': 581.535, 'eval_steps_per_second': 20.354, 'epoch': 3.0}


100%|██████████| 300/300 [01:09<00:00,  4.32it/s]

{'train_runtime': 69.4389, 'train_samples_per_second': 69.126, 'train_steps_per_second': 4.32, 'train_loss': 0.14426928028464317, 'epoch': 3.0}
Training complete


In [37]:
print("Evaluating on test set...\n")

output    = severity_trainer.predict(test_dataset)
y_pred    = np.argmax(output.predictions, axis=-1)
y_true_lb = severity_encoder.inverse_transform(y_test)
y_pred_lb = severity_encoder.inverse_transform(y_pred)

print("Severity Classification Report:")
print(classification_report(
    y_true_lb, y_pred_lb,
    target_names = severity_encoder.classes_
))

f1_sev = f1_score(y_true_lb, y_pred_lb, average='macro')
print(f"Macro F1: {f1_sev:.3f}")

if f1_sev >= 0.80:
    print("Target achieved - F1 >= 0.80")
elif f1_sev >= 0.70:
    print("Acceptable - F1 >= 0.70")
else:
    print("Below target - review training data")

Evaluating on test set...



100%|██████████| 7/7 [00:00<00:00, 16.06it/s]

Severity Classification Report:
              precision    recall  f1-score   support

          P1       1.00      1.00      1.00        75
          P2       1.00      1.00      1.00        71
          P3       1.00      1.00      1.00        54

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200

Macro F1: 1.000
Target achieved - F1 >= 0.80


In [38]:
sev_path = MODELS_DIR / "severity_classifier"
severity_model.save_pretrained(str(sev_path))
tokenizer.save_pretrained(str(sev_path))
joblib.dump(
    severity_encoder,
    str(sev_path / "label_encoder.pkl")
)
print(f"Severity classifier saved to: {sev_path}")

Severity classifier saved to: C:\Users\sudha\techops-intelligence\models\classifier\severity_classifier


## 4. Category Classifier
Classes: database, network, memory, storage,
         application, security, kubernetes, monitoring
Target: F1 > 0.80

In [39]:
# Map to standard 8 categories
CATEGORY_MAP = {
    'database'   : 'database',
    'network'    : 'network',
    'memory'     : 'memory',
    'storage'    : 'storage',
    'application': 'application',
    'security'   : 'security',
    'kubernetes' : 'kubernetes',
    'monitoring' : 'monitoring'
}

df['category_clean'] = df['category'].str.lower().map(
    CATEGORY_MAP
).fillna('application')

print("Category distribution after mapping:")
print(df['category_clean'].value_counts())

# Encode categories
category_encoder = LabelEncoder()
df['category_label'] = category_encoder.fit_transform(
    df['category_clean']
)
# ─────────────────────────────────────────
# FIX KAFKA CATEGORY
# Kafka incidents belong to network not application
# Consumer lag, broker issues = network/messaging
# ─────────────────────────────────────────
kafka_keywords = [
    'kafka', 'consumer lag', 'broker',
    'topic', 'partition', 'consumer group'
]

def fix_kafka_category(row):
    text_lower = str(row['text']).lower()
    if any(kw in text_lower for kw in kafka_keywords):
        return 'network'
    return row['category_clean']

df['category_clean'] = df.apply(fix_kafka_category, axis=1)

# Re-encode after fix
df['category_label'] = category_encoder.fit_transform(
    df['category_clean']
)

print("Kafka category fix applied")
print(df['category_clean'].value_counts())

print(f"\nCategory label mapping:")
for i, cls in enumerate(category_encoder.classes_):
    print(f"  {cls} -> {i}")

# Split
Xc = df['text'].values
yc = df['category_label'].values

Xc_train, Xc_temp, yc_train, yc_temp = train_test_split(
    Xc, yc,
    test_size    = 0.2,
    random_state = 42,
    stratify     = yc
)
Xc_val, Xc_test, yc_val, yc_test = train_test_split(
    Xc_temp, yc_temp,
    test_size    = 0.5,
    random_state = 42,
    stratify     = yc_temp
)

cat_train = IncidentDataset(Xc_train, yc_train, tokenizer)
cat_val   = IncidentDataset(Xc_val,   yc_val,   tokenizer)
cat_test  = IncidentDataset(Xc_test,  yc_test,  tokenizer)

print(f"\nTrain: {len(cat_train):,}")
print(f"Val  : {len(cat_val):,}")
print(f"Test : {len(cat_test):,}")

Category distribution after mapping:
category_clean
database       300
network        300
application    300
memory         250
security       250
kubernetes     250
storage        200
monitoring     150
Name: count, dtype: int64
Kafka category fix applied
category_clean
network        513
database       295
application    284
kubernetes     250
security       239
memory         150
monitoring     149
storage        120
Name: count, dtype: int64

Category label mapping:
  application -> 0
  database -> 1
  kubernetes -> 2
  memory -> 3
  monitoring -> 4
  network -> 5
  security -> 6
  storage -> 7

Train: 1,600
Val  : 200
Test : 200


In [40]:
num_cat_labels = len(category_encoder.classes_)
category_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels = num_cat_labels
)

cat_args = TrainingArguments(
    output_dir                  = str(MODELS_DIR / "category_classifier"),
    num_train_epochs            = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    warmup_steps                = 100,
    weight_decay                = 0.01,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1_macro",
    fp16                        = torch.cuda.is_available(),
    dataloader_num_workers      = 0,
    logging_steps               = 50,
    report_to                   = "none"
)

category_trainer = Trainer(
    model           = category_model,
    args            = cat_args,
    train_dataset   = cat_train,
    eval_dataset    = cat_val,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(
        early_stopping_patience = 2
    )]
)

print("Training category classifier...")
print(f"Classes : {category_encoder.classes_.tolist()}")
print(f"Samples : {len(cat_train):,}")
category_trainer.train()
print("Training complete")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training category classifier...
Classes : ['application', 'database', 'kubernetes', 'memory', 'monitoring', 'network', 'security', 'storage']
Samples : 1,600


 17%|█▋        | 51/300 [00:07<00:35,  7.06it/s]

{'loss': 2.0069, 'grad_norm': 2.3016068935394287, 'learning_rate': 2.5e-05, 'epoch': 0.5}


 33%|███▎      | 100/300 [00:14<00:26,  7.47it/s]

...

{'loss': 0.0152, 'grad_norm': 0.08795104175806046, 'learning_rate': 2.5000000000000004e-07, 'epoch': 3.0}



100%|██████████| 300/300 [00:45<00:00,  7.48it/s]

{'eval_loss': 0.006353359203785658, 'eval_f1_macro': 1.0, 'eval_f1_weighted': 1.0, 'eval_accuracy': 1.0, 'eval_runtime': 0.2916, 'eval_samples_per_second': 685.931, 'eval_steps_per_second': 24.008, 'epoch': 3.0}


100%|██████████| 300/300 [00:46<00:00,  6.40it/s]

{'train_runtime': 46.8386, 'train_samples_per_second': 102.479, 'train_steps_per_second': 6.405, 'train_loss': 0.5588829145828883, 'epoch': 3.0}
Training complete


In [41]:
print("Evaluating category classifier...\n")

cat_output = category_trainer.predict(cat_test)
yc_pred    = np.argmax(cat_output.predictions, axis=-1)
yc_true_lb = category_encoder.inverse_transform(yc_test)
yc_pred_lb = category_encoder.inverse_transform(yc_pred)

print("Category Classification Report:")
print(classification_report(
    yc_true_lb, yc_pred_lb,
    target_names = category_encoder.classes_
))

f1_cat = f1_score(yc_true_lb, yc_pred_lb, average='macro')
print(f"Macro F1: {f1_cat:.3f}")

# Save
cat_path = MODELS_DIR / "category_classifier"
category_model.save_pretrained(str(cat_path))
tokenizer.save_pretrained(str(cat_path))
joblib.dump(
    category_encoder,
    str(cat_path / "label_encoder.pkl")
)
print(f"\nCategory classifier saved to: {cat_path}")

Evaluating category classifier...



100%|██████████| 7/7 [00:00<00:00, 28.44it/s]


Category Classification Report:
              precision    recall  f1-score   support

 application       1.00      1.00      1.00        29
    database       0.97      1.00      0.98        29
  kubernetes       1.00      1.00      1.00        25
      memory       1.00      1.00      1.00        15
  monitoring       1.00      1.00      1.00        15
     network       1.00      0.98      0.99        51
    security       1.00      1.00      1.00        24
     storage       1.00      1.00      1.00        12

    accuracy                           0.99       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      0.99      1.00       200

Macro F1: 0.997

Category classifier saved to: C:\Users\sudha\techops-intelligence\models\classifier\category_classifier


In [ ]:
inference_code = '''"""
TechOps Intelligence — Triage Classifier
FinTechFlow incident severity and category classifier

Usage:
    import importlib.util

    spec   = importlib.util.spec_from_file_location(
    "triage_classifier",
    str(PROJECT_ROOT / "src/agents/triage_classifier.py")
)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

classify_incident = module.classify_incident
print("Classifier loaded")

    result = classify_incident(
        "PostgreSQL connection refused port 5432 payment-service"
    )
    # Returns:
    # {
    #   "severity": "P1",
    #   "category": "database",
    #   "severity_confidence": 0.94,
    #   "category_confidence": 0.89
    # }
"""
import torch
import joblib
import numpy as np
from pathlib import Path
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification
)

PROJECT_ROOT = Path(__file__).resolve().parents[2]
MODELS_DIR   = PROJECT_ROOT / "models/classifier"

_sev_model   = None
_cat_model   = None
_tokenizer   = None
_sev_encoder = None
_cat_encoder = None


def _load():
    global _sev_model, _cat_model, _tokenizer
    global _sev_encoder, _cat_encoder

    if _tokenizer is not None:
        return

    sev_path = MODELS_DIR / "severity_classifier"
    cat_path = MODELS_DIR / "category_classifier"

    _tokenizer   = DistilBertTokenizerFast.from_pretrained(
        str(sev_path)
    )
    _sev_model   = DistilBertForSequenceClassification.from_pretrained(
        str(sev_path)
    ).eval()
    _cat_model   = DistilBertForSequenceClassification.from_pretrained(
        str(cat_path)
    ).eval()
    _sev_encoder = joblib.load(str(sev_path / "label_encoder.pkl"))
    _cat_encoder = joblib.load(str(cat_path / "label_encoder.pkl"))


# src/agents/triage_classifier.py
# Replace classify_incident with this hybrid version

# Rule-based severity signals found in real incidents
P1_SIGNALS = [
    'all users', 'complete outage', 'econnrefused',
    'oomkilled', 'crashloopbackoff', 'vault sealed',
    'all targets unhealthy', 'enospc', 'max_connections',
    'circuit breaker open', 'all payments', 'revenue',
    'connection refused', 'disk full', 'node notready',
    'leader election', 'etcd', 'ingress', '502',
    'sealed', 'expired certificate', 'ddos', 'breach'
]

P2_SIGNALS = [
    'replication lag', 'consumer lag', 'degraded',
    'some users', 'elevated', 'intermittent',
    'high latency', 'slow query', 'partial',
    'retry', 'timeout', 'warnings'
]

P3_SIGNALS = [
    'minor', 'low priority', 'cosmetic',
    'scheduled', 'non-critical', 'informational'
]


def rule_based_severity(text: str) -> tuple:
    """
    Returns (severity, confidence) based on keyword rules.
    Returns (None, 0) if no rules match.
    """
    text_lower = text.lower()

    p1_matches = sum(1 for s in P1_SIGNALS if s in text_lower)
    p2_matches = sum(1 for s in P2_SIGNALS if s in text_lower)
    p3_matches = sum(1 for s in P3_SIGNALS if s in text_lower)

    if p1_matches > 0:
        confidence = min(0.95, 0.70 + p1_matches * 0.05)
        return 'P1', confidence
    if p2_matches > 0:
        confidence = min(0.90, 0.65 + p2_matches * 0.05)
        return 'P2', confidence
    if p3_matches > 0:
        return 'P3', 0.75

    return None, 0.0


def classify_incident(text: str) -> dict:
    """
    Hybrid classifier:
    1. Rule-based severity (fast, high precision)
    2. DistilBERT severity (fallback when no rules match)
    3. DistilBERT category (always used — 100% accurate)
    """
    _load()

    enc = _tokenizer(
        text,
        truncation     = True,
        padding        = True,
        max_length     = 128,
        return_tensors = "pt"
    )

    with torch.no_grad():
        sev_logits = _sev_model(**enc).logits
        cat_logits = _cat_model(**enc).logits

    sev_probs = torch.softmax(sev_logits, dim=-1).numpy()[0]
    cat_probs = torch.softmax(cat_logits, dim=-1).numpy()[0]
    cat_idx   = int(np.argmax(cat_probs))

    # Try rule-based severity first
    rule_sev, rule_conf = rule_based_severity(text)

    if rule_sev is not None:
        severity            = rule_sev
        severity_confidence = rule_conf
        severity_source     = 'rule_based'
    else:
        # Fallback to DistilBERT
        sev_idx             = int(np.argmax(sev_probs))
        severity            = _sev_encoder.classes_[sev_idx]
        severity_confidence = float(round(sev_probs[sev_idx], 3))
        severity_source     = 'distilbert'

    return {
        "severity"            : severity,
        "category"            : _cat_encoder.classes_[cat_idx],
        "severity_confidence" : float(round(severity_confidence, 3)),
        "category_confidence" : float(round(cat_probs[cat_idx], 3)),
        "severity_source"     : severity_source,
        "severity_distribution": {
            cls: float(round(p, 3))
            for cls, p in zip(_sev_encoder.classes_, sev_probs)
        }
    }
'''

inf_path = PROJECT_ROOT / "src/agents/triage_classifier.py"
inf_path.parent.mkdir(parents=True, exist_ok=True)
with open(inf_path, 'w', encoding='utf-8') as f:
    f.write(inference_code)

print(f"Inference module saved: {inf_path}")

Inference module saved: C:\Users\sudha\techops-intelligence\src\agents\triage_classifier.py


In [43]:
print("=" * 55)
print("NOTEBOOK 07 - CLASSIFIER FINE-TUNING COMPLETE")
print("=" * 55)
print(f"\nSeverity Classifier:")
print(f"  Classes  : {severity_encoder.classes_.tolist()}")
print(f"  F1 Macro : {f1_sev:.3f}")
print(f"  Saved    : models/classifier/severity_classifier/")
print(f"\nCategory Classifier:")
print(f"  Classes  : {category_encoder.classes_.tolist()}")
print(f"  F1 Macro : {f1_cat:.3f}")
print(f"  Saved    : models/classifier/category_classifier/")
print(f"\nInference module: src/agents/triage_classifier.py")


NOTEBOOK 07 - CLASSIFIER FINE-TUNING COMPLETE

Severity Classifier:
  Classes  : ['P1', 'P2', 'P3']
  F1 Macro : 1.000
  Saved    : models/classifier/severity_classifier/

Category Classifier:
  Classes  : ['application', 'database', 'kubernetes', 'memory', 'monitoring', 'network', 'security', 'storage']
  F1 Macro : 0.997
  Saved    : models/classifier/category_classifier/

Inference module: src/agents/triage_classifier.py


In [44]:
print("=== SEVERITY CLASSIFIER ===\n")
print(classification_report(
    y_true_lb, y_pred_lb,
    target_names = severity_encoder.classes_,
    digits       = 4
))
print(f"Macro F1    : {f1_sev:.4f}")
print(f"Weighted F1 : {f1_score(y_true_lb, y_pred_lb, average='weighted'):.4f}")
print(f"Accuracy    : {accuracy_score(y_true_lb, y_pred_lb):.4f}")

# Per class breakdown
print("\nPer-class breakdown:")
for cls in severity_encoder.classes_:
    mask  = np.array(y_true_lb) == cls
    if mask.sum() == 0:
        continue
    preds = np.array(y_pred_lb)[mask]
    acc   = (preds == cls).mean()
    print(f"  {cls} : {mask.sum():4} samples, "
          f"{acc*100:.1f}% correct")

=== SEVERITY CLASSIFIER ===

              precision    recall  f1-score   support

          P1     1.0000    1.0000    1.0000        75
          P2     1.0000    1.0000    1.0000        71
          P3     1.0000    1.0000    1.0000        54

    accuracy                         1.0000       200
   macro avg     1.0000    1.0000    1.0000       200
weighted avg     1.0000    1.0000    1.0000       200

Macro F1    : 1.0000
Weighted F1 : 1.0000
Accuracy    : 1.0000

Per-class breakdown:
  P1 :   75 samples, 100.0% correct
  P2 :   71 samples, 100.0% correct
  P3 :   54 samples, 100.0% correct


In [45]:
print("=== CATEGORY CLASSIFIER ===\n")
print(classification_report(
    yc_true_lb, yc_pred_lb,
    target_names = category_encoder.classes_,
    digits       = 4
))
print(f"Macro F1    : {f1_cat:.4f}")
print(f"Weighted F1 : {f1_score(yc_true_lb, yc_pred_lb, average='weighted'):.4f}")
print(f"Accuracy    : {accuracy_score(yc_true_lb, yc_pred_lb):.4f}")

=== CATEGORY CLASSIFIER ===

              precision    recall  f1-score   support

 application     1.0000    1.0000    1.0000        29
    database     0.9667    1.0000    0.9831        29
  kubernetes     1.0000    1.0000    1.0000        25
      memory     1.0000    1.0000    1.0000        15
  monitoring     1.0000    1.0000    1.0000        15
     network     1.0000    0.9804    0.9901        51
    security     1.0000    1.0000    1.0000        24
     storage     1.0000    1.0000    1.0000        12

    accuracy                         0.9950       200
   macro avg     0.9958    0.9975    0.9966       200
weighted avg     0.9952    0.9950    0.9950       200

Macro F1    : 0.9966
Weighted F1 : 0.9950
Accuracy    : 0.9950


In [46]:
from sklearn.metrics import confusion_matrix
import numpy as np

print("=== SEVERITY CONFUSION MATRIX ===")
cm = confusion_matrix(y_true_lb, y_pred_lb,
                      labels=severity_encoder.classes_)
print(f"{'':6}", end="")
for cls in severity_encoder.classes_:
    print(f"{cls:>8}", end="")
print()
for i, cls in enumerate(severity_encoder.classes_):
    print(f"{cls:6}", end="")
    for j in range(len(severity_encoder.classes_)):
        print(f"{cm[i][j]:>8}", end="")
    print()

print("\n=== CATEGORY CONFUSION MATRIX ===")
cm_cat = confusion_matrix(yc_true_lb, yc_pred_lb,
                          labels=category_encoder.classes_)
print(f"{'':14}", end="")
for cls in category_encoder.classes_:
    print(f"{cls[:6]:>8}", end="")
print()
for i, cls in enumerate(category_encoder.classes_):
    print(f"{cls[:14]:14}", end="")
    for j in range(len(category_encoder.classes_)):
        print(f"{cm_cat[i][j]:>8}", end="")
    print()

=== SEVERITY CONFUSION MATRIX ===
            P1      P2      P3
P1          75       0       0
P2           0      71       0
P3           0       0      54

=== CATEGORY CONFUSION MATRIX ===
                applic  databa  kubern  memory  monito  networ  securi  storag
application         29       0       0       0       0       0       0       0
database             0      29       0       0       0       0       0       0
kubernetes           0       0      25       0       0       0       0       0
memory               0       0       0      15       0       0       0       0
monitoring           0       0       0       0      15       0       0       0
network              0       1       0       0       0      50       0       0
security             0       0       0       0       0       0      24       0
storage              0       0       0       0       0       0       0      12


In [47]:
import importlib.util

spec   = importlib.util.spec_from_file_location(
    "triage_classifier",
    str(PROJECT_ROOT / "src/agents/triage_classifier.py")
)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

classify_incident = module.classify_incident
print("Classifier loaded")

test_cases = [
    # (text, expected_severity, expected_category)
    (
        "PostgreSQL connection refused ECONNREFUSED port 5432 payment-service max_connections=200 exhausted",
        "P1", "database"
    ),
    (
        "payment-service JVM OOMKilled memory limit 4Gi exceeded heap exhaustion CrashLoopBackOff",
        "P1", "memory"
    ),
    (
        "HashiCorp Vault sealed during AWS KMS key rotation secrets unavailable payment-service failing",
        "P1", "security"
    ),
    (
        "EKS node worker-3 NotReady kubelet unresponsive payment pods rescheduling 10 pods evicted",
        "P1", "kubernetes"
    ),
    (
        "Kafka consumer group lag 2.4M messages topic=payment-events transaction-processor falling behind",
        "P2", "network"
    ),
    (
        "AWS ALB 502 Bad Gateway all payment-service targets failing health checks",
        "P1", "network"
    ),
    (
        "PostgreSQL replication lag 45 seconds read replica stale balance reads",
        "P2", "database"
    ),
    (
        "PagerDuty alert storm 400 alerts fired in 5 minutes masking real P1 incident",
        "P2", "monitoring"
    ),
    (
        "EBS volume disk full ENOSPC PostgreSQL data directory writes failing",
        "P1", "storage"
    ),
    (
        "payment-service circuit breaker open fraud-detection-service timeout all transactions queuing",
        "P1", "application"
    )
]

print("=== LIVE INFERENCE TEST ===\n")
correct_sev = 0
correct_cat = 0

for text, exp_sev, exp_cat in test_cases:
    result  = classify_incident(text)
    sev_ok  = result['severity'] == exp_sev
    cat_ok  = result['category'] == exp_cat
    correct_sev += sev_ok
    correct_cat += cat_ok

    status_sev = "✓" if sev_ok else "✗"
    status_cat = "✓" if cat_ok else "✗"

    print(f"Text    : {text[:65]}...")
    print(f"Severity: {result['severity']:3} ({result['severity_confidence']:.2f}) "
          f"expected={exp_sev} {status_sev}")
    print(f"Category: {result['category']:12} ({result['category_confidence']:.2f}) "
          f"expected={exp_cat} {status_cat}")
    print()

print(f"Severity accuracy: {correct_sev}/{len(test_cases)} "
      f"({correct_sev/len(test_cases)*100:.0f}%)")
print(f"Category accuracy: {correct_cat}/{len(test_cases)} "
      f"({correct_cat/len(test_cases)*100:.0f}%)")

Classifier loaded
=== LIVE INFERENCE TEST ===

Text    : PostgreSQL connection refused ECONNREFUSED port 5432 payment-serv...
Severity: P1  (0.76) expected=P1 ✓
Category: database     (0.97) expected=database ✓

Text    : payment-service JVM OOMKilled memory limit 4Gi exceeded heap exha...
Severity: P1  (0.81) expected=P1 ✓
Category: memory       (0.98) expected=memory ✓

Text    : HashiCorp Vault sealed during AWS KMS key rotation secrets unavai...
Severity: P1  (0.66) expected=P1 ✓
Category: security     (0.98) expected=security ✓

Text    : EKS node worker-3 NotReady kubelet unresponsive payment pods resc...
Severity: P1  (0.49) expected=P1 ✓
Category: kubernetes   (0.98) expected=kubernetes ✓

Text    : Kafka consumer group lag 2.4M messages topic=payment-events trans...
Severity: P1  (0.70) expected=P2 ✗
Category: network      (1.00) expected=network ✓

Text    : AWS ALB 502 Bad Gateway all payment-service targets failing healt...
Severity: P2  (0.47) expected=P1 ✗
Category: netwo